# Elementary and Core Backend Sweep

This notebook benchmarks arbPlusJAX core interval kernels and elementary helpers against the backends that are available on the current machine.

It separates JAX first-compile time from steady-state execution and includes an optional diagnostics path for JAXPR, HLO, and profiler traces.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "arbplusjax").exists():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate repo root")

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "benchmarks"))

from arbplusjax import jax_diagnostics
import elementary_core_benchmark_support as bench_support

## Environment and Backend Resolution

This section records the exact backend roots, adapter commands, and runtime versions used by the notebook before any sweeps run.


In [2]:
backend_resolution = bench_support.backend_resolution_frame()
bench_support.render_table(backend_resolution)


,item,value
0,repo_root,/home/phili/projects/arbplusJAX
1,output_dir,/home/phili/projects/arbplusJAX/outputs/benchm...
2,ref_prefix,/home/phili/.local/opt/arbplusjax_refs
3,flint_root,/home/phili/.local/opt/arbplusjax_refs/flint/c...
4,boost_root,/home/phili/.local/opt/arbplusjax_refs/boost/c...
5,boost_ref_cmd,/home/phili/projects/arbplusJAX/tools/run_boos...
6,wolfram_linux_dir,/home/phili/Wolfram/14.3/SystemFiles/Kernel/Bi...
7,arb_c_ref_dir,/home/phili/projects/arbplusJAX/stuff/migratio...


In [3]:
runtime_versions = bench_support.runtime_version_frame()
bench_support.render_table(runtime_versions)


,component,version
0,python,3.12.12
1,platform,Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-...
2,machine,x86_64
3,jax,0.8.2
4,jaxlib,0.8.2
5,numpy,2.4.1
6,scipy,1.17.0
7,mpmath,1.3.0
8,pandas,2.3.3
9,jax_default_backend,gpu


In [4]:
diagnostics = jax_diagnostics.JaxDiagnosticsConfig(
    enabled=False,
    capture_jaxpr=False,
    capture_hlo=False,
    trace_execution=False,
)

backend_status = bench_support.backend_status_frame()
bench_support.render_table(backend_status)

,backend,available,detail
0,flint_install,True,/home/phili/.local/opt/arbplusjax_refs/flint/c...
1,arb_flint_c_ref,True,ctypes C reference loader
2,boost,True,/home/phili/projects/arbplusJAX/tools/run_boos...
3,mpmath,True,python module
4,scipy,True,python module
5,jax_scipy,True,python module
6,mathematica,True,"14.3.0 for Linux x86 (64-bit) (July 8, 2025)"


In [5]:
core_results = bench_support.run_core_function_sweep(
    sample_size=4096,
    repeats=8,
    accuracy_sample=128,
    diagnostics=diagnostics,
)
bench_support.render_table(bench_support.summarize_results(core_results))

,suite,function,sample_size,compile_ms,steady_ms_median,recompile_new_shape_ms,peak_rss_delta_mb,mean_width,max_width,scipy_max_rel,mpmath_max_rel,mathematica_max_rel,arb_flint_interval_containment_rate,available_refs
0,core,sinc_pi,4096,213.699650,1.416077,194.708240,0.0000,5.063440e-17,8.881784e-16,2.532101e-16,1.759858e-13,1.199723e+02,NaN,"mathematica,mpmath,scipy"
1,core,sinc,4096,274.273050,1.433034,186.968111,0.0000,1.328830e-16,8.881784e-16,1.638978e-15,2.538604e-16,2.584500e+01,NaN,"mathematica,mpmath,scipy"
2,core,log1p,4096,1738.129978,1.447146,334.477455,0.0000,1.925113e-15,3.552714e-15,2.219464e-16,2.219464e-16,1.570154e-16,NaN,"mathematica,mpmath,scipy"
3,core,tan,4096,221.460110,1.511452,211.512499,0.0000,7.456534e-16,2.664535e-15,2.061186e-16,2.061186e-16,2.061186e-16,0.820312,"mathematica,mpmath,scipy"
4,core,sin_pi,4096,400.794780,1.556008,293.309563,0.0000,4.002861e-15,1.071365e-14,2.137990e-16,2.351994e-14,2.137990e-16,NaN,"mathematica,mpmath,scipy"
5,core,sin,4096,233.380891,1.574242,238.567311,0.0000,5.654305e-16,8.881784e-16,2.207868e-16,2.207868e-16,2.207868e-16,0.796875,"mathematica,mpmath,scipy"
6,core,tan_pi,4096,218.322839,1.583880,225.943403,0.0000,5.667307e-15,5.950795e-14,2.208499e-16,7.203438e-16,2.208499e-16,NaN,"mathematica,mpmath,scipy"
7,core,sqrt,4096,140.896805,1.600835,124.708588,0.0000,5.587944e-15,7.105427e-15,0.000000e+00,0.000000e+00,0.000000e+00,1.000000,"mathematica,mpmath,scipy"
8,core,cos_pi,4096,268.161237,1.641211,291.455816,0.0000,4.190524e-15,1.071365e-14,2.201261e-16,2.631723e-13,2.201261e-16,NaN,"mathematica,mpmath,scipy"
9,core,tanh,4096,138.686432,1.705622,133.845785,0.0000,8.233772e-16,8.881784e-16,1.677005e-16,1.993279e-16,1.993279e-16,0.953125,"mathematica,mpmath,scipy"


In [6]:
mode_results = bench_support.run_core_mode_sweep(
    sample_size=2048,
    repeats=8,
    diagnostics=diagnostics,
)
bench_support.render_table(mode_results)

,function,mode,sample_size,compile_ms,steady_ms_median,recompile_new_shape_ms,peak_rss_delta_mb,mean_width
0,exp,point,2048,53.612994,1.390421,49.204788,0.000000,0.000000e+00
1,exp,basic,2048,1.401731,1.364687,120.291610,0.000000,1.034350e-08
2,exp,adaptive,2048,159.944239,1.587572,174.443463,0.000000,2.068701e-08
3,exp,rigorous,2048,149.301094,1.409549,180.983954,0.000000,2.068701e-08
4,log,point,2048,52.908848,1.619681,56.095833,0.000000,0.000000e+00
...,...,...,...,...,...,...,...,...
59,sinc,rigorous,2048,339.008491,1.684813,289.526883,2.640625,2.702891e-16
60,sinc_pi,point,2048,101.904488,0.182863,115.922857,0.312500,0.000000e+00
61,sinc_pi,basic,2048,2.139583,1.652834,237.616702,0.000000,5.477256e-17
62,sinc_pi,adaptive,2048,345.021106,1.636056,258.708801,3.281250,1.095451e-16


In [7]:
ad_results = bench_support.run_core_ad_sweep(
    sample_size=256,
    repeats=6,
    fd_eps=1e-6,
    diagnostics=diagnostics,
)
bench_support.render_table(ad_results)

,function,mode,sample_size,grad_compile_ms,grad_steady_ms_median,grad_recompile_new_shape_ms,peak_rss_delta_mb,directional_fd_abs_err,directional_fd_rel_err
0,exp,point,256,87.066783,1.586686,83.164785,0.00000,6.139410e-01,1.328056e-08
1,exp,basic,256,326.420102,1.639900,304.873041,0.31250,6.139410e-01,1.328056e-08
2,exp,adaptive,256,384.558818,1.640383,389.169732,0.31250,4.622854e+07,1.000000e+00
3,exp,rigorous,256,365.248878,1.530194,347.602043,0.31250,4.622854e+07,1.000000e+00
4,log,point,256,87.051756,1.715451,76.084332,0.00000,1.021516e-12,2.353379e-12
...,...,...,...,...,...,...,...,...,...
59,sinc,rigorous,256,662.829349,1.494260,616.512689,0.62500,5.127671e-02,1.000000e+00
60,sinc_pi,point,256,176.447953,1.897971,150.128793,0.00000,5.040214e-10,2.492528e-09
61,sinc_pi,basic,256,509.748773,1.624998,525.577310,0.62500,5.040214e-10,2.492528e-09
62,sinc_pi,adaptive,256,664.684471,1.568119,556.965135,1.09375,2.022129e-01,1.000000e+00


In [8]:
elementary_results = bench_support.run_elementary_function_sweep(
    sample_size=2048,
    repeats=8,
    accuracy_sample=128,
    diagnostics=diagnostics,
)
bench_support.render_table(bench_support.summarize_results(elementary_results))

TypeError: float() argument must be a string or a real number, not 'complex'

In [ ]:
core_paths = bench_support.write_result_tables("core_backend_sweep", core_results)
mode_paths = bench_support.write_result_tables("core_mode_sweep", mode_results)
ad_paths = bench_support.write_result_tables("core_ad_sweep", ad_results)
elementary_paths = bench_support.write_result_tables("elementary_backend_sweep", elementary_results)
[
    {"table": "core", **{k: str(v) for k, v in core_paths.items()}},
    {"table": "modes", **{k: str(v) for k, v in mode_paths.items()}},
    {"table": "ad", **{k: str(v) for k, v in ad_paths.items()}},
    {"table": "elementary", **{k: str(v) for k, v in elementary_paths.items()}},
]